# EucherGo vs euchre_zero Visualization

This notebook creates visualizations for the Monte Carlo comparison results between EucherGo and euchre_zero.


In [ ]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")


## Load Data


In [ ]:
# Find and load most recent comparison file
results_dir = project_root / "stats"
comparison_files = list(results_dir.glob("euchergo_vs_euchre_zero_*.json")) if results_dir.exists() else []

if comparison_files:
    latest_file = max(comparison_files, key=lambda p: p.stat().st_mtime)
    print(f"Loading: {latest_file.name}")
    
    with open(latest_file, "r") as f:
        comparison_data = json.load(f)
    
    # Create DataFrame
    games = comparison_data.get("detailed_statistics", {}).get("games", [])
    df = pd.DataFrame(games)
    
    print(f"Loaded {len(df)} games")
else:
    print("No comparison files found. Run euchergo_vs_euchre_zero.py first.")
    comparison_data = None
    df = None


## Win Rate Comparison


In [ ]:
if df is not None:
    euchergo_stats = comparison_data.get("euchergo", {})
    euchre_zero_stats = comparison_data.get("euchre_zero", {})
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    win_rates = [
        euchergo_stats.get('win_rate', 0),
        euchre_zero_stats.get('win_rate', 0)
    ]
    labels = ['EucherGo', 'euchre_zero']
    colors = ['#2ecc71', '#e74c3c']
    
    bars = ax.bar(labels, win_rates, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
    
    # Add value labels on bars
    for bar, rate in zip(bars, win_rates):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{rate:.1%}',
                ha='center', va='bottom', fontsize=14, fontweight='bold')
    
    ax.set_ylabel('Win Rate', fontsize=12)
    ax.set_title('Win Rate Comparison: EucherGo vs euchre_zero', fontsize=14, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")


In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # EucherGo scores
    axes[0].hist(df['euchergo_score'], bins=20, color='#2ecc71', alpha=0.7, edgecolor='black')
    axes[0].axvline(df['euchergo_score'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['euchergo_score'].mean():.2f}")
    axes[0].set_xlabel('Score', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('EucherGo Score Distribution', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # euchre_zero scores
    axes[1].hist(df['euchre_zero_score'], bins=20, color='#e74c3c', alpha=0.7, edgecolor='black')
    axes[1].axvline(df['euchre_zero_score'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df['euchre_zero_score'].mean():.2f}")
    axes[1].set_xlabel('Score', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('euchre_zero Score Distribution', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")


## Tricks Won Comparison


In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Average tricks per game
    avg_tricks = [
        comparison_data.get("euchergo", {}).get('avg_tricks_per_game', 0),
        comparison_data.get("euchre_zero", {}).get('avg_tricks_per_game', 0)
    ]
    
    axes[0].bar(['EucherGo', 'euchre_zero'], avg_tricks, color=['#2ecc71', '#e74c3c'], 
                alpha=0.7, edgecolor='black', linewidth=2)
    axes[0].set_ylabel('Average Tricks per Game', fontsize=12)
    axes[0].set_title('Average Tricks Won per Game', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, (label, val) in enumerate(zip(['EucherGo', 'euchre_zero'], avg_tricks)):
        axes[0].text(i, val, f'{val:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    # Tricks distribution
    axes[1].hist([df['euchergo_tricks'], df['euchre_zero_tricks']], 
                 bins=15, label=['EucherGo', 'euchre_zero'], 
                 color=['#2ecc71', '#e74c3c'], alpha=0.6, edgecolor='black')
    axes[1].set_xlabel('Tricks Won', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('Tricks Won Distribution', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")


## Score vs Tricks Correlation


In [ ]:
if df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # EucherGo: Score vs Tricks
    axes[0].scatter(df['euchergo_tricks'], df['euchergo_score'], 
                    color='#2ecc71', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    axes[0].set_xlabel('Tricks Won', fontsize=12)
    axes[0].set_ylabel('Final Score', fontsize=12)
    axes[0].set_title('EucherGo: Score vs Tricks', fontsize=14, fontweight='bold')
    
    # Add correlation coefficient
    corr = df['euchergo_tricks'].corr(df['euchergo_score'])
    axes[0].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                transform=axes[0].transAxes, fontsize=11,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[0].grid(alpha=0.3)
    
    # euchre_zero: Score vs Tricks
    axes[1].scatter(df['euchre_zero_tricks'], df['euchre_zero_score'], 
                    color='#e74c3c', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    axes[1].set_xlabel('Tricks Won', fontsize=12)
    axes[1].set_ylabel('Final Score', fontsize=12)
    axes[1].set_title('euchre_zero: Score vs Tricks', fontsize=14, fontweight='bold')
    
    # Add correlation coefficient
    corr = df['euchre_zero_tricks'].corr(df['euchre_zero_score'])
    axes[1].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                transform=axes[1].transAxes, fontsize=11,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")


## Cumulative Win Rate Over Time


In [ ]:
if df is not None and 'euchergo_team' in df.columns:
    # Calculate cumulative win rates
    euchergo_wins = (df['winner'] == df['euchergo_team']).cumsum()
    euchre_zero_wins = (df['winner'] == (1 - df['euchergo_team'])).cumsum()
    total_games = np.arange(1, len(df) + 1)
    
    euchergo_cumulative_rate = euchergo_wins / total_games
    euchre_zero_cumulative_rate = euchre_zero_wins / total_games
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(total_games, euchergo_cumulative_rate, label='EucherGo', color='#2ecc71', linewidth=2)
    ax.plot(total_games, euchre_zero_cumulative_rate, label='euchre_zero', color='#e74c3c', linewidth=2)
    ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='50% (Even)')
    
    ax.set_xlabel('Game Number', fontsize=12)
    ax.set_ylabel('Cumulative Win Rate', fontsize=12)
    ax.set_title('Cumulative Win Rate Over Time', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)
    ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")


## Hands Played Distribution


In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist(df['hands_played'], bins=range(df['hands_played'].min(), df['hands_played'].max() + 2), 
           color='#3498db', alpha=0.7, edgecolor='black')
    ax.axvline(df['hands_played'].mean(), color='red', linestyle='--', linewidth=2, 
              label=f"Mean: {df['hands_played'].mean():.2f}")
    ax.axvline(df['hands_played'].median(), color='orange', linestyle='--', linewidth=2, 
              label=f"Median: {df['hands_played'].median():.0f}")
    
    ax.set_xlabel('Hands Played', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('Distribution of Hands Played per Game', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")


## Comprehensive Comparison Dashboard


In [ ]:
if df is not None:
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    euchergo_stats = comparison_data.get("euchergo", {})
    euchre_zero_stats = comparison_data.get("euchre_zero", {})
    
    # 1. Win Rate
    ax1 = fig.add_subplot(gs[0, 0])
    win_rates = [euchergo_stats.get('win_rate', 0), euchre_zero_stats.get('win_rate', 0)]
    ax1.bar(['EucherGo', 'euchre_zero'], win_rates, color=['#2ecc71', '#e74c3c'], alpha=0.7)
    ax1.set_ylabel('Win Rate')
    ax1.set_title('Win Rate')
    ax1.set_ylim([0, 1])
    for i, (label, val) in enumerate(zip(['EucherGo', 'euchre_zero'], win_rates)):
        ax1.text(i, val, f'{val:.1%}', ha='center', va='bottom', fontweight='bold')
    
    # 2. Average Score
    ax2 = fig.add_subplot(gs[0, 1])
    avg_scores = [euchergo_stats.get('avg_score', 0), euchre_zero_stats.get('avg_score', 0)]
    ax2.bar(['EucherGo', 'euchre_zero'], avg_scores, color=['#2ecc71', '#e74c3c'], alpha=0.7)
    ax2.set_ylabel('Average Score')
    ax2.set_title('Average Score per Game')
    for i, (label, val) in enumerate(zip(['EucherGo', 'euchre_zero'], avg_scores)):
        ax2.text(i, val, f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
    
    # 3. Average Tricks
    ax3 = fig.add_subplot(gs[0, 2])
    avg_tricks = [euchergo_stats.get('avg_tricks_per_game', 0), euchre_zero_stats.get('avg_tricks_per_game', 0)]
    ax3.bar(['EucherGo', 'euchre_zero'], avg_tricks, color=['#2ecc71', '#e74c3c'], alpha=0.7)
    ax3.set_ylabel('Average Tricks')
    ax3.set_title('Average Tricks per Game')
    for i, (label, val) in enumerate(zip(['EucherGo', 'euchre_zero'], avg_tricks)):
        ax3.text(i, val, f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
    
    # 4. Score Distribution
    ax4 = fig.add_subplot(gs[1, :2])
    ax4.hist([df['euchergo_score'], df['euchre_zero_score']], 
            bins=15, label=['EucherGo', 'euchre_zero'], 
            color=['#2ecc71', '#e74c3c'], alpha=0.6, edgecolor='black')
    ax4.set_xlabel('Final Score')
    ax4.set_ylabel('Frequency')
    ax4.set_title('Score Distribution')
    ax4.legend()
    
    # 5. Tricks Distribution
    ax5 = fig.add_subplot(gs[1, 2])
    ax5.hist([df['euchergo_tricks'], df['euchre_zero_tricks']], 
            bins=15, label=['EucherGo', 'euchre_zero'], 
            color=['#2ecc71', '#e74c3c'], alpha=0.6, edgecolor='black')
    ax5.set_xlabel('Tricks Won')
    ax5.set_ylabel('Frequency')
    ax5.set_title('Tricks Distribution')
    ax5.legend()
    
    # 6. Cumulative Win Rate
    if 'euchergo_team' in df.columns:
        euchergo_wins = (df['winner'] == df['euchergo_team']).cumsum()
        total_games = np.arange(1, len(df) + 1)
        euchergo_cumulative_rate = euchergo_wins / total_games
        
        ax6 = fig.add_subplot(gs[2, :])
        ax6.plot(total_games, euchergo_cumulative_rate, label='EucherGo', color='#2ecc71', linewidth=2)
        ax6.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        ax6.set_xlabel('Game Number')
        ax6.set_ylabel('Cumulative Win Rate')
        ax6.set_title('Cumulative Win Rate Over Time')
        ax6.legend()
        ax6.set_ylim([0, 1])
        ax6.grid(alpha=0.3)
    
    plt.suptitle('EucherGo vs euchre_zero Comprehensive Comparison', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.show()
else:
    print("No data available for visualization.")
